# S5 : Spark et PySpark, en pratique

**Question de Salma :** « Le top des marques a pris presque deux minutes sur le cluster. Demain, je veux cinq analyses, avec des filtres différents. Faut-il écrire cinq jobs MapReduce ? »

Un seul notebook pour toute la séance, **une seule session Spark** : ne redémarrez pas le noyau entre deux parties. Exécutez les cellules dans l'ordre et **prévoyez le résultat avant chaque exécution**.

| Partie | Horaire | Contenu |
|---|---|---|
| 1. Démarrer | 0:00 à 0:45 | session, lecture, RDD et DataFrame, transformations et actions, plan, interface |
| 2. Transformer et agréger | 0:45 à 1:30 | colonnes calculées, agrégations, taux de conversion, le mois d'août comparé à S4 |
| 3. Interroger et ranger | 1:40 à 2:25 | Spark SQL, schéma typé, jointures, Parquet partitionné, mesures |
| 4. Optimiser et conclure | 2:25 à 3:00 | tâches et vagues, shuffle, cache, AQE, fonctions de fenêtre, réponse à Salma |

| Adresse | Ce que vous y voyez |
|---|---|
| <http://localhost:8080> | le master : deux workers, puis votre application |
| <http://localhost:4040> | votre application : Jobs, Stages, SQL, Executors |

Avant de commencer, à la racine du dépôt : `docker compose --profile spark up -d`, puis `python scripts/verifier_env.py --spark`.

# Partie 1 : démarrer

## Étape 2 : ouvrir la session

Le noyau Python de ce notebook est le **driver**. En ouvrant la session, il demande au master 4 cœurs ; le master fait lancer un executor de 2 cœurs sur chacun des deux workers.

- `spark.sql.shuffle.partitions = 4` : 4 partitions après un shuffle, au lieu de 200 (assez pour nos données).
- `spark.sql.adaptive.enabled = false` : l'exécution adaptative est coupée pour lire le plan tel quel (partie 4).

L'adresse du master n'est pas dans le code : le conteneur `jupyter-spark` la fournit (variable `PYSPARK_SUBMIT_ARGS` de `compose.yaml`). Sans elle, Spark démarre en `local[*]`, sur une seule machine.

In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
         .appName("Qissaria-S5")
         .config("spark.sql.shuffle.partitions", "4")
         .config("spark.sql.adaptive.enabled", "false")
         .getOrCreate())
print(spark.version, spark.sparkContext.master)

**Ce que vous devez voir :** `4.2.0 spark://spark-master:7077`. Sur <http://localhost:8080>, l'application `Qissaria-S5` est *RUNNING* avec 4 cœurs.

La cellule suivante définit deux petites fonctions d'observation, utilisées aux étapes 4 et 6.

In [ ]:
sc = spark.sparkContext


def jobs():
    """Nombre de jobs lancés depuis l'ouverture de la session."""
    return len(sc.statusTracker().getJobIdsForGroup(None))


def resume_jobs(depuis=0, combien=3):
    """Les jobs numérotés à partir de depuis (combien au plus) : leurs stages et leurs tâches."""
    suivi = sc.statusTracker()
    for job_id in sorted(j for j in suivi.getJobIdsForGroup(None) if j >= depuis)[:combien]:
        info = suivi.getJobInfo(job_id)
        taches = [suivi.getStageInfo(s).numTasks for s in sorted(info.stageIds) if suivi.getStageInfo(s)]
        print(f"job {job_id} : {len(info.stageIds)} stage(s), tâches {taches}")


print("Jobs lancés jusqu'ici :", jobs())

## Étape 3 : lire un DataFrame

Le fichier a les neuf colonnes du clickstream d'août : `event_time`, `event_type`, `product_id`, `category_id`, `category_code`, `brand`, `price`, `user_id`, `user_session`.

Chaque executor lit **lui-même** sa partie du fichier : le chemin doit exister sur toutes les machines. Ici, le dépôt est monté en `/home/jovyan/work` dans les quatre conteneurs. Sur un vrai cluster, on lit HDFS (partie 2).

In [ ]:
CHEMIN = "file:///home/jovyan/work/data/echantillon/clickstream_petit.csv"
df = spark.read.option("header", True).csv(CHEMIN)
df.printSchema()
df.select("event_time", "event_type", "brand", "user_id").show(5, truncate=False)
df.count()

**Q1.** Quel est le type de `price` ? Que devient la marque vide de la 5e ligne ? Où se trouvent les 18 lignes pendant le calcul ?

*La correction est projetée juste après l'exercice. Notez ici ce qui vous a surpris.*

## RDD et DataFrame : le même comptage

Un **RDD** est la collection distribuée d'origine de Spark : ici, des lignes de texte, sans colonnes. Le programme ci-dessous est le MapReduce de S4 écrit en Python : `map` produit `(clé, 1)`, `reduceByKey` additionne après le shuffle.

In [ ]:
lignes = sc.textFile(CHEMIN)
entete = lignes.first()
types = (lignes
  .filter(lambda l: l != entete)
  .map(lambda l: (l.split(",")[1], 1))
  .reduceByKey(lambda a, b: a + b))
sorted(types.collect())

In [ ]:
(df.groupBy("event_type")
   .count()
   .orderBy("event_type")
   .show())

**Ce que vous devez voir :** les deux versions donnent `cart` 3, `purchase` 6, `view` 9.

| | RDD | DataFrame |
|---|---|---|
| Une ligne, c'est | une chaîne, puis un tuple : `l.split(",")[1]` | des colonnes nommées : `"event_type"` |
| Le plan | celui que vous écrivez, pas à pas | réécrit par l'optimiseur, qui ne lit que les colonnes utiles |
| Votre code tourne | dans des processus Python, à côté des executors (aller-retour JVM ↔ Python) | dans la JVM des executors |
| Un piège | `split(",")` casse si un champ contient une virgule | le lecteur CSV gère les guillemets |

## Exercice (6 min) : les clients les plus actifs

Voici un programme RDD. Exécutez-le, puis :

1. Quelle colonne porte l'indice 7 ? Que compte ce programme ?
2. Écrivez la version DataFrame dans la cellule suivante : même tri (compte décroissant, puis identifiant), trois lignes.
3. Prévoyez le résultat sur les 18 événements **avant** d'exécuter.

Aide : `groupBy`, `count`, `orderBy(F.desc("count"), F.asc(...))`, `limit`.

In [ ]:
par_client = (lignes.filter(lambda l: l != entete)
                    .map(lambda l: (l.split(",")[7], 1))
                    .reduceByKey(lambda a, b: a + b))
par_client.sortBy(lambda kv: (-kv[1], kv[0])).take(3)

In [ ]:
# Votre version DataFrame (trois lignes, même tri)

*La correction est projetée juste après l'exercice. Notez ici ce qui vous a surpris.*

## Étape 4 : compter les jobs

Une **transformation** (`filter`, `select`, `groupBy(...).count()`, `orderBy`) décrit un nouveau DataFrame : rien n'est calculé. Une **action** (`show`, `count`, `collect`, une écriture) demande un résultat et lance au moins un **job**.

**Exercice (5 min).** La cellule suivante affiche, après chaque ligne, le nombre de jobs lancés depuis son début. **Avant de l'exécuter**, écrivez vos six prévisions. Quelles lignes sont des actions ? Une ligne vous surprendra : laquelle ?

In [ ]:
debut_etape4 = depart = jobs()     # les jobs sont numérotés à partir de 0 : le prochain porte ce numéro
df = spark.read.option("header", True).csv(CHEMIN)              # (1)
print("(1) read.csv          :", jobs() - depart)
achats = df.filter(F.col("event_type") == "purchase")           # (2)
print("(2) filter            :", jobs() - depart)
par_marque = achats.groupBy("brand").count()                    # (3)
print("(3) groupBy().count() :", jobs() - depart)
par_marque.explain()                                            # (4)
print("(4) explain           :", jobs() - depart)
par_marque.orderBy(F.desc("count")).show()                      # (5)
print("(5) show              :", jobs() - depart)
print("    count() renvoie   :", par_marque.count())            # (6)
print("(6) count             :", jobs() - depart)

*La correction est projetée juste après l'exercice. Notez ici ce qui vous a surpris.*

In [ ]:
# La même lecture avec un schéma fourni, puis avec inferSchema : combien de jobs ?
SCHEMA = ("event_time STRING, event_type STRING, product_id STRING, category_id STRING, "
          "category_code STRING, brand STRING, price STRING, user_id STRING, user_session STRING")
depart = jobs(); spark.read.option("header", True).schema(SCHEMA).csv(CHEMIN)
print("schéma fourni :", jobs() - depart, "job(s)")
depart = jobs(); spark.read.option("header", True).option("inferSchema", True).csv(CHEMIN)
print("inferSchema   :", jobs() - depart, "job(s)")

## Étape 5 : lire le plan

`explain()` affiche le plan physique ; il se lit **de bas en haut**.

| Ligne du plan | Ce qu'elle fait | En S4 |
|---|---|---|
| `FileScan csv` | lit le fichier, mais seulement 2 colonnes sur 9 (`ReadSchema`) | lecture du bloc |
| `Filter`, `Project` | garde les achats, puis la marque | le mapper |
| `HashAggregate … partial_count` | compte par marque dans chaque partition | le combiner |
| `Exchange hashpartitioning(brand, 4)` | envoie chaque marque à l'une des 4 partitions | le **shuffle** |
| `HashAggregate … count` | additionne les comptes partiels | le reducer |

Un `Exchange` est une **frontière de stage** : le job de `show` a donc deux stages.

In [ ]:
par_marque.explain()
print("Partitions de lecture     :", df.rdd.getNumPartitions())
print("Partitions après Exchange :", par_marque.rdd.getNumPartitions())

**Ce que vous devez voir :** le plan du tableau, puis `1` partition de lecture (le petit fichier forme une seule partition) et `4` partitions après l'`Exchange` (le réglage de la session). Un stage a autant de **tâches** que de partitions.

## Étape 6 : l'interface de l'application

Ouvrir <http://localhost:4040> (la session de ce notebook doit rester ouverte).

1. **Jobs** : retrouver les jobs de l'étape 4 (`csv`, `showString`, `count`).
2. Cliquer sur `showString`, puis *DAG Visualization* : deux stages séparés par l'`Exchange`.
3. **Stages** : lire la colonne *Tasks* de ces deux stages.
4. **Executors** : le driver et les deux executors, avec leurs cœurs et leurs tâches.

**Exercice (3 min).** Sans regarder : combien de stages et combien de tâches pour le job `count` de la ligne (6) ? Vérifiez dans l'onglet Jobs, ou avec la cellule ci-dessous.

In [ ]:
resume_jobs(debut_etape4)   # les trois jobs de l'étape 4, dans l'ordre : (1) read.csv, (5) show, (6) count

*La correction est projetée juste après l'exercice. Notez ici ce qui vous a surpris.*

## Ce qu'il faut retenir de la partie 1

- Une **application** Spark : un driver (ce notebook) et des executors réservés pour toute la session.
- Le driver découpe chaque action en **job**, puis en **stages** (frontière : le shuffle), puis en **tâches** (une par partition).
- Un **DataFrame** a un schéma : Spark optimise le plan et ne lit que les colonnes utiles. Le RDD reste le MapReduce écrit en Python.
- Les **transformations** décrivent, les **actions** calculent. Sans cache, chaque action repart de la source.
- Pour vérifier : `explain()` avant, l'interface 4040 pendant et après.

# Partie 2 : transformer et agréger

**Question de Salma :** « Le top des marques, c'est fait. Je veux aussi le chiffre d'affaires par rayon, les clients par jour, et savoir quelles marques transforment les visites en achats. Sur tout le mois d'août. »

La session et `df` (les 18 événements) de la partie 1 servent encore.

## Étape 7 : de nouvelles colonnes

`withColumn` ajoute une colonne calculée par une **expression** : rien n'est calculé tout de suite, Spark ajoute un `Project` au plan. La fonction Python `enrichir` ne fait qu'assembler des expressions Spark : elle resservira sur le mois d'août.

- `cast("double")` : le prix devient un nombre (une valeur illisible deviendrait `NULL`) ;
- `F.split(..., r"\.")` : découpe selon une expression régulière ; le point s'écrit `\.` ;
- `F.to_date(F.substring(...))` : les 10 premiers caractères de `event_time`, en date.

In [ ]:
def enrichir(d):        # une fonction Python qui assemble des expressions Spark
    return (d.withColumn("prix",  F.col("price").cast("double"))
             .withColumn("rayon", F.split("category_code", r"\.").getItem(0))
             .withColumn("jour",  F.to_date(F.substring("event_time", 1, 10))))


ev = enrichir(df)
ev.select("event_type", "brand", "price", "prix", "rayon", "jour").show(5)
ev.printSchema()

**Ce que vous devez voir :** `prix` en `double` (`1200.0`), `rayon` = `electronics`, `apparel`, `home` ou `kids`, `jour` en `date` (`2026-08-01`). `price` reste une chaîne : `df` n'a pas changé, `ev` est un nouveau DataFrame.

### Expression ou UDF ?

Une UDF est une fonction Python appliquée à chaque valeur. Elle marche, mais les lignes passent, par lots, de la JVM à un processus Python puis reviennent, et l'optimiseur ne voit qu'une boîte noire. Dans le plan, `ArrowEvalPython` signale l'UDF (`BatchEvalPython` dans les versions antérieures à Spark 4).

In [ ]:
@F.udf("string")
def rayon_udf(code):
    if code is None:
        return None
    return code.split(".")[0]


avec_udf = ev.withColumn("r", rayon_udf("category_code"))
avec_udf.groupBy("r").count().orderBy("r").show()
avec_udf.select("r").explain()

## Exercice (5 min) : les gammes de prix

1. Ajoutez à `ev` une colonne `gamme` : `haut` à partir de 1 000 DH, `milieu` à partir de 300 DH, `entrée` sinon (`F.when(...).when(...).otherwise(...)`).
2. Comptez les événements par gamme, du plus grand au plus petit.
3. Prévoyez le résultat **avant** d'exécuter : novatek coûte 1 200 DH, sahara 800, atlas 350, le jouet sans marque 90.

Question : que se passerait-il si l'on écrivait d'abord `.when(F.col("prix") >= 300, "milieu")` ?

In [ ]:
# Votre programme

*La correction est projetée juste après l'exercice. Notez ici ce qui vous a surpris.*

## Étape 8 : plusieurs mesures à la fois

`groupBy(...).agg(...)` calcule plusieurs mesures en un seul passage. `F.count("*")` compte les lignes, `F.count("brand")` les valeurs non nulles, `F.countDistinct` les valeurs différentes (le plus coûteux : il faut garder les valeurs).

In [ ]:
achats = ev.filter((F.col("event_type") == "purchase") & F.col("brand").isNotNull())
(achats.groupBy("brand")
       .agg(F.count("*").alias("achats"),
            F.sum("prix").alias("chiffre"),
            F.avg("prix").alias("prix_moyen"),
            F.countDistinct("user_id").alias("clients"))
       .orderBy(F.desc("chiffre")).show())

**Ce que vous devez voir :** novatek 3 achats, 3600.0 DH, 3 clients ; sahara 1, 800.0 ; atlas 2, 700.0. Par nombre d'achats, atlas est 2e ; par chiffre d'affaires, c'est sahara.

Le lien avec S4 : une moyenne ne pouvait pas servir de combiner. Dans le plan ci-dessous, Spark calcule `partial_avg` (somme et nombre par partition), puis `avg`.

In [ ]:
achats.groupBy("brand").agg(F.avg("prix")).explain()

## Étape 9 : le taux de conversion

`pivot` met une colonne par valeur de `event_type`. Donner la liste des valeurs évite un job de plus (Spark devrait sinon chercher les valeurs distinctes).

In [ ]:
conversion = (ev.filter(F.col("brand").isNotNull())
  .groupBy("brand")
  .pivot("event_type", ["view", "cart", "purchase"])   # 3 colonnes
  .count()
  .fillna(0)                                            # pas de panier : 0
  .withColumn("conversion", F.round(F.col("purchase") / F.col("view"), 2)))
conversion.orderBy("brand").show()

**Ce que vous devez voir :** atlas 3 vues, 2 paniers, 2 achats, 0.67 ; novatek 4, 1, 3, 0.75 ; sahara 1, 0, 1, 1.0.

## Exercice (5 min) : les marques mises au panier

Affichez les marques renseignées qui ont **au moins deux** événements `cart`, triées par marque. Quel filtre vient **avant** `groupBy` ? Lequel vient **après** ? Prévoyez le résultat.

Aide : après `groupBy("brand").count()`, la colonne s'appelle `count`.

In [ ]:
# Votre programme

*La correction est projetée juste après l'exercice. Notez ici ce qui vous a surpris.*

## Étape 10 : le mois d'août

Le même programme, sur les 1,4 Go du clickstream d'août (`data/clickstream_aout.csv`, sur la clé USB ou généré par `scripts/generer_clickstream.py`). Si votre cluster HDFS de S3 tourne (`docker compose --profile hadoop up -d`), vous pouvez lire la copie de HDFS à la place.

**Prévoyez d'abord :** Spark découpe la lecture en morceaux d'au plus 128 Mo. Combien de partitions, donc combien de tâches au premier stage ?

In [ ]:
import time

AOUT = "file:///home/jovyan/work/data/clickstream_aout.csv"
# AOUT = "hdfs://namenode:8020/qissaria/clickstream/clickstream_aout.csv"   # avec HDFS
aout = spark.read.option("header", True).csv(AOUT)
print("Partitions de lecture :", aout.rdd.getNumPartitions())


def chrono(nom, resultat, lignes=10):
    """Affiche un résultat, puis la durée de l'action qui le calcule."""
    debut = time.perf_counter()
    resultat.show(lignes, truncate=False)
    print(f"{nom} : {time.perf_counter() - debut:.1f} s")


chrono("Top 10 des marques achetées",
       aout.filter((F.col("event_type") == "purchase") & F.col("brand").isNotNull())
           .groupBy("brand").count()
           .orderBy(F.desc("count"), F.asc("brand")).limit(10))

**Ce que vous devez voir :** **12** partitions (1,4 Go / 128 Mo ≈ 11,2) : ce sont les 12 blocs HDFS de S3. Puis les mêmes comptes que votre job MapReduce de S4 : novatek 10 080, kasbah 9 301, orbita 8 907, medina 8 550, sahara 8 497, dunes 8 448, zellij 8 243, kora 8 049, tafilalt 8 023, tidra 7 953.

Notez votre durée Spark à côté de celle de votre job S4.

## Étape 11 : cinq analyses, une seule application

Les cinq analyses de Salma, l'une après l'autre, dans la même session. Chacune est un job de la même application (onglet Jobs de <http://localhost:4040>).

In [ ]:
ev_aout = enrichir(aout)
marque = F.col("brand").isNotNull()

chrono("1. Événements par type", ev_aout.groupBy("event_type").count().orderBy("event_type"))
chrono("2. Top 5 des marques achetées",
       ev_aout.filter((F.col("event_type") == "purchase") & marque)
              .groupBy("brand").count().orderBy(F.desc("count"), "brand").limit(5))
chrono("3. Top 5 des marques consultées",
       ev_aout.filter((F.col("event_type") == "view") & marque)
              .groupBy("brand").count().orderBy(F.desc("count"), "brand").limit(5))
chrono("4. Chiffre d'affaires par rayon",
       ev_aout.filter(F.col("event_type") == "purchase").groupBy("rayon")
              .agg(F.count("*").alias("achats"), F.round(F.sum("prix")).cast("long").alias("chiffre"))
              .orderBy(F.desc("chiffre")).limit(5))
chrono("5. Clients distincts par jour",
       ev_aout.groupBy("jour").agg(F.countDistinct("user_id").alias("clients")).orderBy("jour").limit(5))

**Ce que vous devez voir :** view 10 280 795, cart 514 288, purchase 178 620 ; `electronics` 52 567 achats pour 342 547 783 DH ; 37 082 clients distincts le 1er août.

Un seul démarrage (étape 2) pour les cinq analyses. Mais chacune relit les 12 partitions du fichier : la partie 4 gardera `ev_aout` en mémoire avec le **cache**.

## Exercice (6 min) : les marques qui convertissent

Sur `ev_aout` : pour les 5 marques qui ont le plus de vues, affichez les vues, les achats et le taux de conversion **en pourcentage** (achats / vues × 100, deux décimales). Quelle marque convertit le moins bien parmi les cinq ? Mesurez la durée avec `chrono`.

In [ ]:
# Votre programme

*La correction est projetée juste après l'exercice. Notez ici ce qui vous a surpris.*

## Ce qu'il faut retenir de la partie 2

- `withColumn` ajoute une colonne calculée par une **expression** (`cast`, `F.split`, `F.when`) ; une UDF seulement si aucune fonction n'existe.
- `groupBy(...).agg(...)` calcule plusieurs mesures en un seul passage ; `pivot` met une colonne par valeur.
- Un filtre **avant** `groupBy` garde des lignes, un filtre **après** garde des groupes.
- Sur le mois d'août : les **mêmes comptes** que le job MapReduce de S4, 12 partitions comme les 12 blocs HDFS, une seule application pour toutes les analyses.

**Ne fermez pas la session pendant la pause.**

# Partie 3 : interroger et ranger

**Question de Salma :** « Mon équipe parle SQL, pas Python. Je veux aussi le nom des produits et le responsable de chaque rayon. Et le rapport du matin doit se relire en une seconde, pas en dix. »

## Étape 12 : la même question en SQL

`createOrReplaceTempView` donne un nom de table à un DataFrame, le temps de la session (aucune donnée n'est copiée). `spark.sql(...)` renvoie un DataFrame : rien n'est calculé avant une action.

In [ ]:
ev.createOrReplaceTempView("evenements")       # un nom de table pour SQL
top_sql = spark.sql("""
    SELECT brand, COUNT(*) AS achats
    FROM evenements
    WHERE event_type = 'purchase' AND brand IS NOT NULL
    GROUP BY brand
    ORDER BY achats DESC, brand
    LIMIT 10""")
top_sql.show()

### SQL et DataFrame : un seul moteur

`exceptAll` renvoie les lignes d'un DataFrame absentes de l'autre (doublons compris). Deux `0` : les résultats sont identiques. Comparez ensuite les deux plans.

In [ ]:
top_df = (ev.filter((F.col("event_type") == "purchase") & F.col("brand").isNotNull())
            .groupBy("brand").agg(F.count("*").alias("achats"))
            .orderBy(F.desc("achats"), "brand").limit(10))
print(top_sql.exceptAll(top_df).count(), top_df.exceptAll(top_sql).count())
top_sql.explain()
top_df.explain()

**Ce que vous devez voir :** `0 0`, puis deux plans identiques (`TakeOrderedAndProject`, `HashAggregate`, `Exchange`, `FileScan csv [event_type,brand]`). SQL ou DataFrame : une question de lisibilité, pas de vitesse.

## Exercice (4 min) : WHERE et HAVING

Réécrivez en SQL l'exercice de la partie 2 : les marques renseignées avec **au moins deux** événements `cart`, triées par marque. Quelle condition va dans `WHERE`, laquelle dans `HAVING` ? Pourquoi ne peut-on pas écrire `WHERE COUNT(*) >= 2` ?

In [ ]:
# Votre requête : spark.sql(""" ... """).show()

*La correction est projetée juste après l'exercice. Notez ici ce qui vous a surpris.*

## Étape 13 : un schéma explicite

Vous donnez noms et types : Spark n'a rien à deviner (aucun job à la lecture), et l'argent est compté exactement en `DECIMAL(10,2)`. Un `double` arrondit en binaire : `0.1 + 0.2` vaut `0.30000000000000004`.

In [ ]:
SCHEMA = ("event_time TIMESTAMP, event_type STRING, product_id INT, "
          "category_id BIGINT, category_code STRING, brand STRING, "
          "price DECIMAL(10,2), user_id INT, user_session STRING")
typ = (spark.read.option("header", True)
            .option("timestampFormat", "yyyy-MM-dd HH:mm:ss z")   # 2026-08-01 10:00:00 UTC
            .schema(SCHEMA).csv(CHEMIN))
typ.printSchema()
typ.filter("event_type = 'purchase'").agg(F.sum("price")).show()
print(0.1 + 0.2)

**Ce que vous devez voir :** `event_time: timestamp`, `product_id: integer`, `price: decimal(10,2)` ; la somme des achats vaut `5100.00`.

## Étape 14 : joindre le catalogue

`data/echantillon/catalogue_petit.csv` donne le libellé de trois produits. Le produit `1004` (le jouet sans marque) n'y est pas.

In [ ]:
cat = spark.read.option("header", True).csv("file:///home/jovyan/work/data/echantillon/catalogue_petit.csv")
print(ev.join(cat, "product_id").count(),            # jointure interne
      ev.join(cat, "product_id", "left").count())    # jointure gauche
(ev.join(cat, "product_id", "left")
   .filter(F.col("libelle").isNull())
   .select("product_id", "event_type", "brand", "libelle").show())

**Ce que vous devez voir :** `17 18`. La jointure **interne** perd l'événement du produit `1004` ; la jointure **gauche** le garde, avec `libelle` à `NULL`. Pour un rapport, on garde les événements (jointure gauche), puis on compte les `NULL`.

## Exercice (5 min) : le piège de la clé dupliquée

Un collègue envoie `data/echantillon/catalogue_doublon.csv`. Joignez `ev` avec ce fichier (jointure gauche sur `product_id`), puis :

1. Comptez les lignes après la jointure. Combien en attendiez-vous ?
2. Calculez achats et chiffre d'affaires par marque : qu'est-ce qui a changé par rapport à l'étape 8 ?
3. Trouvez la cause **dans le catalogue**, avec un `groupBy`. Proposez une correction.

Le contrôle à retenir : une jointure gauche sur une clé **unique** garde exactement le nombre de lignes de gauche.

In [ ]:
dbl = spark.read.option("header", True).csv("file:///home/jovyan/work/data/echantillon/catalogue_doublon.csv")
# Votre programme

*La correction est projetée juste après l'exercice. Notez ici ce qui vous a surpris.*

## Étape 15 : le rapport par rayon, sur août

`data/echantillon/rayons.csv` donne, pour chacun des 10 rayons, son libellé, son responsable et son taux de marge. Cette petite table est **diffusée** (*broadcast*) : copiée sur chaque executor, elle évite de faire traverser le réseau aux 11 millions de lignes. Cherchez `BroadcastHashJoin` dans le plan.

In [ ]:
rayons = spark.read.option("header", True).csv("file:///home/jovyan/work/data/echantillon/rayons.csv")
rapport = (ev_aout.filter(F.col("event_type") == "purchase")
  .join(rayons, "rayon", "left")
  .groupBy("rayon", "libelle", "responsable")
  .agg(F.round(F.sum("prix")).cast("long").alias("chiffre"),
       F.round(F.sum(F.col("prix") * F.col("marge"))).cast("long").alias("marge"))
  .orderBy(F.desc("chiffre")))
rapport.explain()
chrono("Rapport par rayon", rapport, 12)

**Ce que vous devez voir :** `electronics` (Yassine) 342 547 783 DH de chiffre, 27 403 823 DH de marge ; puis `computers` et `appliances`. Une ligne `NULL` : 5 615 721 DH d'achats sans catégorie, que la jointure gauche a gardés.

## Étape 16 : écrire en Parquet, partitionné par jour

Parquet range les données **colonne par colonne**, en binaire, avec leurs types, compressées. `partitionBy("jour")` crée un dossier par jour. Chaque executor écrit ses fichiers lui-même : `sorties/` est monté en écriture dans les quatre conteneurs. L'écriture prend environ une minute.

In [ ]:
SORTIE = "file:///home/jovyan/work/sorties/clickstream_aout_parquet"
# SORTIE = "hdfs://namenode:8020/qissaria/parquet/clickstream_aout"   # avec HDFS
debut = time.perf_counter()
(ev_aout.select("event_time", "event_type", "product_id", "brand",
                "prix", "user_id", "rayon", "jour")
        .repartition("jour")          # un fichier par jour, pas 12
        .write.mode("overwrite")
        .partitionBy("jour")          # un dossier par jour
        .parquet(SORTIE))
print(f"Écriture : {time.perf_counter() - debut:.0f} s")

In [ ]:
import os

dossier = "/home/jovyan/work/sorties/clickstream_aout_parquet"
taille = sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(dossier) for f in fs)
jours = sorted(d for d in os.listdir(dossier) if d.startswith("jour="))
print(len(jours), "dossiers :", jours[0], "...", jours[-1])
print(f"Parquet : {taille / 1e6:.0f} Mo ; CSV : {os.path.getsize('/home/jovyan/work/data/clickstream_aout.csv') / 1e6:.0f} Mo")

**Ce que vous devez voir :** 30 dossiers, de `jour=2026-08-01` à `jour=2026-08-30`, et un Parquet environ **douze fois plus petit** que le CSV (environ 125 Mo contre 1 500 Mo).

## Étape 17 : relire et filtrer

In [ ]:
pq = spark.read.parquet(SORTIE)
pq.printSchema()                                  # les types sont revenus
le_15 = (pq.filter(F.col("jour") == "2026-08-15")
           .filter(F.col("event_type") == "purchase")
           .groupBy("rayon").agg(F.round(F.sum("prix")).cast("long").alias("chiffre"))
           .orderBy(F.desc("chiffre")))
le_15.explain()
le_15.show(5)

**Ce que vous devez voir :** `prix: double` et `jour: date` sans schéma ni `cast` ; dans le plan, `PartitionFilters: [... jour = 2026-08-15]` (un dossier lu sur 30), `ReadSchema` avec 3 colonnes, `Batched: true`. Le 15 août : `electronics` 11 328 010 DH, `computers` 5 002 690 DH.

## Exercice (6 min) : mesurer honnêtement

Avec la fonction `mesure` ci-dessous (une exécution d'échauffement, puis trois mesures, médiane), comparez CSV et Parquet sur deux questions :

1. la somme des prix du mois (sur le CSV, avec `cast`) ;
2. le nombre d'événements du 15 août (sur le CSV : `event_time` commence par `2026-08-15` ; sur le Parquet : `jour`).

Avant de lancer : quel rapport de vitesse prévoyez-vous, et pourquoi le second écart est-il plus grand ?

In [ ]:
import statistics


def mesure(nom, action, repetitions=3):
    """Une exécution d'échauffement, puis la médiane de plusieurs mesures."""
    action()
    durees = []
    for _ in range(repetitions):
        debut = time.perf_counter()
        action()
        durees.append(time.perf_counter() - debut)
    print(f"{nom} : {[round(d, 1) for d in durees]} s, médiane {statistics.median(durees):.1f} s")


csv_aout = spark.read.option("header", True).csv(AOUT)
mesure("CSV, somme des prix", lambda: csv_aout.agg(F.sum(F.col("price").cast("double"))).collect())
# Vos trois autres mesures

*La correction est projetée juste après l'exercice. Notez ici ce qui vous a surpris.*

## Ce qu'il faut retenir de la partie 3

- `spark.sql(...)` et l'API DataFrame produisent le **même plan** : `exceptAll` le vérifie, `explain` le montre.
- `WHERE` filtre des lignes, `HAVING` des groupes.
- Un **schéma explicite** donne les bons types (`DECIMAL` pour l'argent) sans lecture préalable.
- Jointure **gauche** pour garder les événements ; vérifier l'**unicité de la clé** et compter les lignes avant et après. Une petite table est diffusée (*broadcast*).
- **Parquet** partitionné : écrit une fois, relu vite (colonnes utiles, dossiers sautés, types conservés).

# Partie 4 : optimiser et conclure

**Question de Salma :** « Pourquoi certaines analyses prennent dix secondes et d'autres une demi-seconde ? Et comment accélérer le rapport du matin sans changer le résultat ? »

La fonction `stages` interroge l'API de l'interface 4040 (la même chose que l'onglet Stages) : pour chaque stage des jobs lancés depuis `depuis`, le nombre de tâches, les octets lus et les octets écrits pour le shuffle.

In [ ]:
import json
import urllib.request

API = f"http://127.0.0.1:4040/api/v1/applications/{sc.applicationId}"


def stages(depuis):
    """Les stages terminés des jobs numérotés à partir de depuis."""
    liste_jobs = json.load(urllib.request.urlopen(API + "/jobs"))
    par_id = {s["stageId"]: s for s in json.load(urllib.request.urlopen(API + "/stages"))}
    for job in sorted(liste_jobs, key=lambda j: j["jobId"]):
        if job["jobId"] < depuis:
            continue
        for sid in sorted(job["stageIds"]):
            s = par_id.get(sid)
            if s and s["status"] == "COMPLETE":
                print(f"job {job['jobId']}, stage {sid} : {s['numTasks']} tâches, "
                      f"{s['inputBytes'] / 1e6:.0f} Mo lus, shuffle écrit {s['shuffleWriteBytes'] / 1e3:.0f} Ko, "
                      f"temps cumulé des tâches {s['executorRunTime'] / 1000:.1f} s")

## Étape 18 : prévoir les tâches

**Exercice (3 min), avant d'exécuter.** Le fichier d'août est lu en 12 partitions ; l'application a 2 executors de 2 cœurs. Combien de tâches au premier stage ? Combien en même temps ? Combien de **vagues** ? Si une tâche dure 3 s, combien dure le stage ?

Après l'exécution, ouvrez l'onglet **Stages** de <http://localhost:4040>, le premier stage, puis *Event Timeline*.

In [ ]:
depuis = jobs()
chrono("Top des marques",
       ev_aout.filter((F.col("event_type") == "purchase") & F.col("brand").isNotNull())
              .groupBy("brand").count().orderBy(F.desc("count")).limit(10))
stages(depuis)

*La correction est projetée juste après l'exercice. Notez ici ce qui vous a surpris.*

### Le shuffle : ce qui traverse le réseau

Comparez le shuffle du top des marques (cellule précédente) avec celui-ci.

In [ ]:
depuis = jobs()
chrono("Clients distincts par jour",
       ev_aout.groupBy("jour").agg(F.countDistinct("user_id").alias("clients")))
stages(depuis)

**Ce que vous devez voir :** quelques Ko de shuffle pour le top des marques (le comptage partiel réduit chaque partition à une ligne par marque), plusieurs Mo pour les clients distincts (il faut envoyer les couples jour, client). Les trois réflexes : filtrer **tôt**, agréger **tôt**, regrouper sur des clés **peu nombreuses**.

## Étape 19 : garder un résultat en mémoire

`.cache()` est une transformation : c'est la **première action** qui remplit le cache. Les actions suivantes partent de la mémoire des executors (`InMemoryTableScan` dans le plan). Pendant ce temps, ouvrez l'onglet **Storage**.

In [ ]:
achats_aout = ev_aout.filter(F.col("event_type") == "purchase").cache()
chrono("Remplir le cache", achats_aout.groupBy().count())       # 1re action
chrono("Marques", achats_aout.groupBy("brand").count())
chrono("Rayons",  achats_aout.groupBy("rayon").agg(F.sum("prix")))
chrono("Jours",   achats_aout.groupBy("jour").count())
achats_aout.groupBy("brand").count().explain()

**Ce que vous devez voir :** environ 12 s pour remplir le cache (lire 1,5 Go, garder 178 620 achats), puis environ **0,3 s** par analyse au lieu de 10. Onglet Storage : 12 partitions, environ 21 Mo, *Disk Memory Deserialized*. Libérez ensuite la mémoire :

In [ ]:
achats_aout.unpersist()

## Étape 20 : l'exécution adaptative (AQE)

Depuis le début, la session utilise 4 partitions après un shuffle et coupe l'exécution adaptative. Revenons à la valeur par défaut de Spark (200 partitions), sans puis avec AQE. L'écriture au format `noop` calcule tout le résultat sans rien écrire : une seule action, un seul job à observer.

In [ ]:
spark.conf.set("spark.sql.shuffle.partitions", "200")    # la valeur par défaut
for aqe in ("false", "true"):
    spark.conf.set("spark.sql.adaptive.enabled", aqe)
    depuis = jobs()
    debut = time.perf_counter()
    pq.groupBy("rayon").agg(F.sum("prix")).write.format("noop").mode("overwrite").save()
    print(f"AQE {aqe} : {time.perf_counter() - debut:.1f} s")
    stages(depuis)
spark.conf.set("spark.sql.shuffle.partitions", "4")      # retour aux réglages
spark.conf.set("spark.sql.adaptive.enabled", "false")    # de la séance

**Ce que vous devez voir :** sans AQE, 5 tâches de lecture puis **200 tâches** pour 11 rayons, presque toutes vides ; avec AQE, 5 tâches puis **1 seule**. Après chaque shuffle, AQE regarde les tailles réelles : il regroupe les petites partitions, choisit une jointure diffusée si une table est petite, découpe une partition trop grosse. Elle est activée par défaut depuis Spark 3.2.

## Exercice final (10 min) : le top 3 de chaque rayon

Sur le Parquet d'août (`pq`), pour chaque rayon renseigné, les **3 produits les plus achetés** : rayon, `product_id`, marque, nombre d'achats, rang.

1. Comptez les achats par (rayon, produit, marque).
2. Numérotez les produits **dans chaque rayon** avec une fonction de fenêtre : `w = Window.partitionBy("rayon").orderBy(F.desc("achats"), "product_id")`, puis `F.row_number().over(w)`.
3. Gardez les rangs 1 à 3, triez par rayon puis rang.
4. Lisez le plan : combien d'`Exchange` ? À quoi sert chacun ?

Un `groupBy` donnerait une ligne par rayon ; une fenêtre garde toutes les lignes et leur ajoute un rang.

In [ ]:
from pyspark.sql import Window

# Votre programme

*La correction est projetée juste après l'exercice. Notez ici ce qui vous a surpris.*

## Réponse à Salma

« Faut-il écrire cinq jobs MapReduce pour cinq analyses ? »

- **Non.** Une seule application Spark, ouverte une fois : chaque analyse est un job. Sur août, chacune prend environ 10 s, avec **les mêmes comptes** que le job MapReduce de S4.
- Les analyses s'écrivent en **DataFrame** ou en **SQL** : même moteur, même plan.
- Le rapport du matin lit un **Parquet partitionné par jour** : moins d'une seconde au lieu de dix.
- Plusieurs analyses sur le même sous-ensemble : un **cache**, puis environ 0,3 s chacune.
- Avant de croire un chiffre : **vérifier** (`exceptAll`, nombre de lignes avant et après une jointure, unicité des clés).

## Ticket de sortie (sur papier)

1. Pourquoi `df.groupBy("brand").count()` ne lance-t-il aucun job ?
2. Que signale un `Exchange` dans un plan ? Donnez un exemple de la séance.
3. Pourquoi une jointure gauche peut-elle **augmenter** le nombre de lignes ?
4. Deux raisons pour lesquelles Parquet est plus rapide que CSV pour le rapport du matin.

## Fin de séance

Fermez la session, puis, à la racine du dépôt : `docker compose --profile spark stop` (rien n'est effacé).

In [ ]:
spark.stop()